 
 # 01 · Publicación del Catálogo de Reglas DQ Silver — YelpConnect

 **Entregable:** E5 · Data Quality  
 **Tipo:** Publicación de catálogo gobernado  
 **Capa evaluada:** Silver  
 **Tabla destino:** `workspace.yelp_gov.dq_reglas`  
 **Ejecución:** Idempotente mediante `MERGE`

 ---

 ## Propósito

 Publicar y mantener el catálogo gobernado de reglas Data Quality
 aplicables a la capa Silver de YelpConnect.

 La infraestructura física debe haber sido creada previamente por:

 `00_setup/02_configurar_modelo_gobierno`

 ---

 ## Principio arquitectónico

 **Bronze conserva · Silver conforma · DQ valida · Trust & Safety evalúa señales**

 Data Quality y Trust & Safety son responsabilidades diferentes:

 - DQ evalúa confiabilidad y calidad del dato.
 - T&S evalúa patrones heurísticos sobre datos conformados.
 - Una falla DQ no equivale a una señal T&S.
 - Una señal T&S no confirma fraude.

 ---

 ## Responsabilidad

 Este notebook:

 - valida que exista `workspace.yelp_gov.dq_reglas`;
 - define las 22 reglas Silver;
 - publica las reglas mediante `MERGE`;
 - utiliza `regla_id` como identificador estable;
 - valida el catálogo publicado.

 Este notebook **NO**:

 - crea schemas;
 - crea tablas de gobierno;
 - ejecuta las reglas DQ;
 - genera `dq_resultados`;
 - transforma datos Silver;
 - calcula señales Trust & Safety.

 ---

 ## Contrato de salida

 Tabla:

 `workspace.yelp_gov.dq_reglas`

 Grano:

 **1 fila = 1 regla Data Quality**


In [0]:
# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"
GOV_SCHEMA = "yelp_gov"

DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)

LAYER = "silver"

EXPECTED_RULE_COUNT = 22


print("=" * 72)
print("YELPCONNECT - CATÁLOGO DATA QUALITY SILVER")
print("=" * 72)

print(
    f"Tabla destino    : "
    f"{DQ_RULE_TABLE}"
)

print(
    f"Capa             : "
    f"{LAYER}"
)

print(
    f"Reglas esperadas : "
    f"{EXPECTED_RULE_COUNT}"
)

print("=" * 72)

# COMMAND ----------


 ## 1 · Precondición

 Debe existir previamente:

 `workspace.yelp_gov.dq_reglas`

 Esta tabla es creada por:

 `00_setup/02_configurar_modelo_gobierno`

 Si no existe, el pipeline está siendo ejecutado fuera de secuencia.

In [0]:
# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLA DE CATÁLOGO
# ============================================================

if not spark.catalog.tableExists(
    DQ_RULE_TABLE
):

    raise Exception(
        f"""
        ❌ No existe la tabla:

        {DQ_RULE_TABLE}

        Ejecutar primero:

        00_setup/
        02_configurar_modelo_gobierno
        """
    )


print(
    f"✔ Catálogo disponible: "
    f"{DQ_RULE_TABLE}"
)


In [0]:
# COMMAND ----------

# ============================================================
# 4. VALIDAR CONTRATO
# ============================================================

required_columns = {
    "regla_id",
    "capa",
    "tabla_objetivo",
    "columna",
    "dimension",
    "tipo_regla",
    "descripcion",
    "expresion_sql",
    "criticidad",
    "umbral_pct"
}


actual_columns = {
    field.name
    for field in spark.table(
        DQ_RULE_TABLE
    ).schema.fields
}


missing_columns = sorted(
    required_columns
    -
    actual_columns
)


if missing_columns:

    raise Exception(
        "❌ dq_reglas no cumple "
        "el contrato esperado. "
        "Faltan columnas: "
        +
        ", ".join(
            missing_columns
        )
    )


print(
    "✔ Contrato de dq_reglas validado."
)

# COMMAND ----------

 ## 2 · Dimensiones Data Quality

 Las reglas Silver cubren:

 - **Completitud:** existencia de valores obligatorios.
 - **Unicidad:** ausencia de identificadores duplicados.
 - **Validez:** valores dentro de dominios esperados.
 - **Consistencia:** coherencia lógica de métricas.
 - **Integridad:** referencias válidas entre entidades.
 - **Actualidad:** coherencia temporal respecto al procesamiento.

 Las entidades evaluadas son:

 - Review
 - Business
 - User
 - Tip
 - Check-in

# COMMAND ----------

 
 ## 3 · Consideración especial — DQ-SL-012

 La regla `DQ-SL-012` monitorea:

 `Review.user_id → User.user_id`

 El dataset `User` utilizado en el proyecto corresponde a una
 **muestra parcial**.

 Por ello, la ausencia de un `user_id` de Review dentro de `slv_user`:

 - no invalida automáticamente la Review;
 - no representa fraude;
 - no constituye una señal Trust & Safety;
 - representa una limitación esperada de cobertura del dataset.

 La regla utiliza:

 - criticidad: `media`
 - umbral: `90%`

 Esto permite monitorear cobertura sin bloquear injustificadamente
 el pipeline.


In [0]:
# COMMAND ----------

# ============================================================
# 5. SCHEMA EXPLÍCITO
# ============================================================

schema_reglas = StructType([

    StructField(
        "regla_id",
        StringType(),
        False
    ),

    StructField(
        "capa",
        StringType(),
        False
    ),

    StructField(
        "tabla_objetivo",
        StringType(),
        False
    ),

    StructField(
        "columna",
        StringType(),
        True
    ),

    StructField(
        "dimension",
        StringType(),
        False
    ),

    StructField(
        "tipo_regla",
        StringType(),
        False
    ),

    StructField(
        "descripcion",
        StringType(),
        False
    ),

    StructField(
        "expresion_sql",
        StringType(),
        True
    ),

    StructField(
        "criticidad",
        StringType(),
        False
    ),

    StructField(
        "umbral_pct",
        DoubleType(),
        False
    )
])

# COMMAND ----------

 ## 4 · Reglas Silver

 Este refactor mantiene sin cambios:

 - `regla_id`;
 - condiciones;
 - tablas objetivo;
 - criticidades;
 - umbrales.

 La mejora realizada es únicamente estructural:

 **configuración física y publicación del catálogo quedan separadas.**

In [0]:

# COMMAND ----------

# ============================================================
# 6. CATÁLOGO DE REGLAS SILVER
# ============================================================

reglas_silver = [

    # ========================================================
    # REVIEW
    # ========================================================

    (
        "DQ-SL-001",
        "silver",
        "workspace.yelp_silver.slv_review",
        "stars",
        "validez",
        "CONDITION",
        "El rating de una reseña debe estar entre 1 y 5",
        "stars BETWEEN 1 AND 5",
        "alta",
        100.0
    ),

    (
        "DQ-SL-002",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_id",
        "unicidad",
        "UNIQUENESS",
        "review_id debe ser único dentro de Silver",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-SL-003",
        "silver",
        "workspace.yelp_silver.slv_review",
        "business_id",
        "integridad",
        "CONDITION",
        "Toda Review debe referenciar un Business existente",
        """
        business_id IN (
            SELECT business_id
            FROM workspace.yelp_silver.slv_business
        )
        """,
        "alta",
        100.0
    ),

    (
        "DQ-SL-004",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_id",
        "completitud",
        "CONDITION",
        "review_id no debe ser nulo",
        "review_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-005",
        "silver",
        "workspace.yelp_silver.slv_review",
        "business_id",
        "completitud",
        "CONDITION",
        "business_id no debe ser nulo",
        "business_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-006",
        "silver",
        "workspace.yelp_silver.slv_review",
        "user_id",
        "completitud",
        "CONDITION",
        "user_id no debe ser nulo",
        "user_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-007",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_text",
        "completitud",
        "CONDITION",
        "El texto de la reseña no debe ser nulo ni vacío",
        """
        review_text IS NOT NULL
        AND LENGTH(TRIM(review_text)) > 0
        """,
        "media",
        99.0
    ),

    (
        "DQ-SL-008",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_ts",
        "validez",
        "CONDITION",
        "La fecha de la reseña debe poder representarse como timestamp",
        "review_ts IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-009",
        "silver",
        "workspace.yelp_silver.slv_review",
        "useful",
        "consistencia",
        "CONDITION",
        "El contador useful no puede ser negativo",
        "useful IS NULL OR useful >= 0",
        "media",
        100.0
    ),

    (
        "DQ-SL-010",
        "silver",
        "workspace.yelp_silver.slv_review",
        "funny",
        "consistencia",
        "CONDITION",
        "El contador funny no puede ser negativo",
        "funny IS NULL OR funny >= 0",
        "media",
        100.0
    ),

    (
        "DQ-SL-011",
        "silver",
        "workspace.yelp_silver.slv_review",
        "cool",
        "consistencia",
        "CONDITION",
        "El contador cool no puede ser negativo",
        "cool IS NULL OR cool >= 0",
        "media",
        100.0
    ),

    (
        "DQ-SL-012",
        "silver",
        "workspace.yelp_silver.slv_review",
        "user_id",
        "integridad",
        "CONDITION",
        "El user_id de Review debería estar presente en la muestra User",
        """
        user_id IN (
            SELECT user_id
            FROM workspace.yelp_silver.slv_user
        )
        """,
        "media",
        90.0
    ),

    (
        "DQ-SL-013",
        "silver",
        "workspace.yelp_silver.slv_review",
        "review_ts",
        "actualidad",
        "CONDITION",
        "La fecha de la reseña no puede ser posterior a su procesamiento Silver",
        """
        review_ts <= _silver_processed_at
        """,
        "media",
        100.0
    ),

    # ========================================================
    # BUSINESS
    # ========================================================

    (
        "DQ-SL-014",
        "silver",
        "workspace.yelp_silver.slv_business",
        "business_id",
        "completitud",
        "CONDITION",
        "business_id no debe ser nulo",
        "business_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-015",
        "silver",
        "workspace.yelp_silver.slv_business",
        "business_id",
        "unicidad",
        "UNIQUENESS",
        "business_id debe ser único",
        None,
        "alta",
        100.0
    ),

    (
        "DQ-SL-016",
        "silver",
        "workspace.yelp_silver.slv_business",
        "estrellas",
        "validez",
        "CONDITION",
        "El rating agregado del Business debe estar entre 0 y 5",
        """
        estrellas IS NULL
        OR estrellas BETWEEN 0 AND 5
        """,
        "media",
        100.0
    ),

    (
        "DQ-SL-017",
        "silver",
        "workspace.yelp_silver.slv_business",
        "n_resenas",
        "consistencia",
        "CONDITION",
        "El número de reseñas del Business no puede ser negativo",
        """
        n_resenas IS NULL
        OR n_resenas >= 0
        """,
        "media",
        100.0
    ),

    # ========================================================
    # USER
    # ========================================================

    (
        "DQ-SL-018",
        "silver",
        "workspace.yelp_silver.slv_user",
        "user_id",
        "completitud",
        "CONDITION",
        "user_id no debe ser nulo",
        "user_id IS NOT NULL",
        "alta",
        100.0
    ),

    (
        "DQ-SL-019",
        "silver",
        "workspace.yelp_silver.slv_user",
        "user_id",
        "unicidad",
        "UNIQUENESS",
        "user_id debe ser único",
        None,
        "alta",
        100.0
    ),

    # ========================================================
    # TIP
    # ========================================================

    (
        "DQ-SL-020",
        "silver",
        "workspace.yelp_silver.slv_tip",
        "business_id",
        "integridad",
        "CONDITION",
        "Todo Tip debe referenciar un Business existente",
        """
        business_id IN (
            SELECT business_id
            FROM workspace.yelp_silver.slv_business
        )
        """,
        "media",
        100.0
    ),

    # ========================================================
    # CHECK-IN
    # ========================================================

    (
        "DQ-SL-021",
        "silver",
        "workspace.yelp_silver.slv_checkin_event",
        "business_id",
        "integridad",
        "CONDITION",
        "Todo Check-in debe referenciar un Business existente",
        """
        business_id IN (
            SELECT business_id
            FROM workspace.yelp_silver.slv_business
        )
        """,
        "media",
        100.0
    ),

    (
        "DQ-SL-022",
        "silver",
        "workspace.yelp_silver.slv_checkin_event",
        "checkin_ts",
        "validez",
        "CONDITION",
        "Todo evento Check-in debe poseer un timestamp válido",
        "checkin_ts IS NOT NULL",
        "media",
        100.0
    )
]

# COMMAND ----------

# ============================================================
# 7. CREAR DATAFRAME
# ============================================================

df_reglas_silver = (
    spark.createDataFrame(
        reglas_silver,
        schema=schema_reglas
    )
)


defined_rule_count = (
    df_reglas_silver.count()
)


print(
    f"✔ Reglas Silver definidas: "
    f"{defined_rule_count}"
)


if defined_rule_count != EXPECTED_RULE_COUNT:

    raise Exception(
        f"""
        ❌ Cantidad inesperada de reglas.

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Definidas:
        {defined_rule_count}
        """
    )


display(
    df_reglas_silver
    .orderBy(
        "regla_id"
    )
)

# COMMAND ----------

# ============================================================
# 8. VALIDAR regla_id ÚNICO
# ============================================================

duplicate_rule_ids = (

    df_reglas_silver

    .groupBy(
        "regla_id"
    )

    .count()

    .filter(
        "count > 1"
    )

    .count()
)


if duplicate_rule_ids > 0:

    raise Exception(
        f"""
        ❌ Existen {duplicate_rule_ids}
        regla_id duplicados.
        """
    )


print(
    "✔ regla_id únicos."
)

# COMMAND ----------

# ============================================================
# 9. VISTA TEMPORAL
# ============================================================

df_reglas_silver.createOrReplaceTempView(
    "src_dq_reglas_silver"
)


print(
    "✔ Vista temporal: "
    "src_dq_reglas_silver"
)

# COMMAND ----------

 
 

## 5 · Publicación idempotente

 El catálogo se publica mediante:

 `MERGE INTO`

 utilizando:

 `regla_id`

 como clave estable.

 Si la regla existe, se actualiza.

 Si no existe, se inserta.

 Esto permite re-ejecutar el notebook sin generar duplicados.


In [0]:

# COMMAND ----------

# ============================================================
# 10. MERGE IDEMPOTENTE
# ============================================================

spark.sql(
    f"""
    MERGE INTO
        {DQ_RULE_TABLE} AS target

    USING
        src_dq_reglas_silver AS source

    ON
        target.regla_id =
        source.regla_id

    WHEN MATCHED THEN

        UPDATE SET

            target.capa =
                source.capa,

            target.tabla_objetivo =
                source.tabla_objetivo,

            target.columna =
                source.columna,

            target.dimension =
                source.dimension,

            target.tipo_regla =
                source.tipo_regla,

            target.descripcion =
                source.descripcion,

            target.expresion_sql =
                source.expresion_sql,

            target.criticidad =
                source.criticidad,

            target.umbral_pct =
                source.umbral_pct

    WHEN NOT MATCHED THEN

        INSERT
        (
            regla_id,
            capa,
            tabla_objetivo,
            columna,
            dimension,
            tipo_regla,
            descripcion,
            expresion_sql,
            criticidad,
            umbral_pct
        )

        VALUES
        (
            source.regla_id,
            source.capa,
            source.tabla_objetivo,
            source.columna,
            source.dimension,
            source.tipo_regla,
            source.descripcion,
            source.expresion_sql,
            source.criticidad,
            source.umbral_pct
        )
    """
)


print(
    "✔ Reglas Silver registradas/actualizadas."
)

# COMMAND ----------

 


 ## 6 · Validación del catálogo

 Al finalizar deben existir:

 **22 reglas Silver**

 dentro de:

 `workspace.yelp_gov.dq_reglas`

 Las reglas de otras capas no participan en esta validación.


In [0]:

# COMMAND ----------

# ============================================================
# 11. VALIDAR TOTAL
# ============================================================

total_silver = (

    spark.table(
        DQ_RULE_TABLE
    )

    .filter(
        "LOWER(capa) = 'silver'"
    )

    .count()
)


print("=" * 72)
print("CATÁLOGO DATA QUALITY - SILVER")
print("=" * 72)

print(
    f"Total reglas Silver: "
    f"{total_silver}"
)

print("=" * 72)


if total_silver != EXPECTED_RULE_COUNT:

    raise Exception(
        f"""
        ❌ Cantidad inesperada.

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Encontradas:
        {total_silver}
        """
    )


print(
    "✔ Cantidad de reglas validada."
)

# COMMAND ----------

# ============================================================
# 12. DETALLE DEL CATÁLOGO
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            regla_id,
            tabla_objetivo,
            columna,
            dimension,
            tipo_regla,
            criticidad,
            umbral_pct,
            descripcion

        FROM
            {DQ_RULE_TABLE}

        WHERE
            LOWER(capa) = 'silver'

        ORDER BY
            regla_id
        """
    )
)

# COMMAND ----------

# ============================================================
# 13. RESUMEN POR DIMENSIÓN
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            dimension,

            COUNT(*) AS total_reglas

        FROM
            {DQ_RULE_TABLE}

        WHERE
            LOWER(capa) = 'silver'

        GROUP BY
            dimension

        ORDER BY
            dimension
        """
    )
)

# COMMAND ----------

# ============================================================
# 14. RESUMEN POR TABLA
# ============================================================

display(
    spark.sql(
        f"""
        SELECT

            tabla_objetivo,

            COUNT(*) AS total_reglas

        FROM
            {DQ_RULE_TABLE}

        WHERE
            LOWER(capa) = 'silver'

        GROUP BY
            tabla_objetivo

        ORDER BY
            tabla_objetivo
        """
    )
)

# COMMAND ----------

 ## Definition of Done

 `01_publicar_catalogo_reglas_dq_silver` se considera completado cuando:

 - [ ] `workspace.yelp_gov.dq_reglas` existe;
 - [ ] el contrato esperado está disponible;
 - [ ] existen 22 reglas Silver definidas;
 - [ ] no existen `regla_id` duplicados;
 - [ ] el `MERGE` finaliza correctamente;
 - [ ] existen exactamente 22 registros con `capa = 'silver'`;
 - [ ] las dimensiones DQ esperadas están representadas;
 - [ ] no se ejecutaron reglas;
 - [ ] no se generaron resultados DQ;
 - [ ] no se modificaron tablas Silver.

 ---

 ## Siguiente etapa

 ```text
 01_publicar_catalogo_reglas_dq_silver
         ↓
 02_publicar_catalogo_reglas_ts
         ↓
 01_ingestar_bronze
 ```

In [0]:
# COMMAND ----------

# ============================================================
# 15. RESUMEN FINAL
# ============================================================

print()
print("=" * 72)
print("CATÁLOGO DQ SILVER PUBLICADO")
print("=" * 72)

print(
    f"Tabla  : "
    f"{DQ_RULE_TABLE}"
)

print(
    f"Capa   : "
    f"{LAYER}"
)

print(
    f"Reglas : "
    f"{total_silver}"
)

print()
print(
    "✔ Publicación idempotente completada."
)

print(
    "✔ No se ejecutaron reglas DQ."
)

print(
    "✔ No se modificaron datos Silver."
)

print("=" * 72)